In [ ]:
# стирание базы полное до нуля

In [29]:
!docker stop neo4j_travel || true
!docker rm neo4j_travel || true
!docker volume rm neo4j_travel_data || true

!docker run -d --name neo4j_travel \
  -p 7475:7474 -p 7688:7687 \
  -e NEO4J_AUTH=neo4j/12345678 \
  -v neo4j_travel_data:/data \
  neo4j:5

# !docker ps --filter name=neo4j_travel

neo4j_travel
neo4j_travel
neo4j_travel_data
95a4ce4395681ec1e149136551b92091d49b40b41f4f2e967c39ec8c21f411f9


In [1]:
# сборка промпта и запрос к DeepSeek - выход ABox в виде JSON


In [30]:
import os
import json
import requests

PROMPT_PATH = "prompt.txt"
MSG_PATH = "message.txt"
ONTO_PATH = "ontology.txt"

DEEPSEEK_API_KEY = "sk-8da8031877dc4c39ae8d7b624c70f01f"
if not DEEPSEEK_API_KEY:
    raise RuntimeError("Set env var DEEPSEEK_API_KEY first (export DEEPSEEK_API_KEY=...).")

DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")

with open(PROMPT_PATH, "r", encoding="utf-8") as f:
    prompt_template = f.read()

with open(MSG_PATH, "r", encoding="utf-8") as f:
    message_text = f.read().strip()

with open(ONTO_PATH, "r", encoding="utf-8") as f:
    ontology_text = f.read().strip()

prompt = (prompt_template
          .replace("{RDF_ONTOLOGY_HERE}", ontology_text)
          .replace("{TEXT_HERE}", message_text))

url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
    "Content-Type": "application/json",
}
payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "Return ONLY JSON. No markdown. Follow the template strictly."},
        {"role": "user", "content": prompt},
    ],
    "temperature": 0.0,
}

resp = requests.post(url, headers=headers, json=payload, timeout=180)
resp.raise_for_status()
data = resp.json()

# DeepSeek API finish reason (transport-level)
api_finish = data["choices"][0].get("finish_reason", "")
content = data["choices"][0]["message"]["content"]

# Save to variable for next cells
answer = content

emoji = "🟢" if api_finish != "length" else "🔴"
print(f"{emoji} API finish_reason = {api_finish}")
print(answer)  # preview

🟢 API finish_reason = stop
{
  "payload": {
    "ontology_version": "https://example.org/pcfr#",
    "nodes": [
      {
        "label": "Encounter",
        "key": "encounter_1",
        "properties": {
          "sourcePostID": 1,
          "encounterDurationSec": 10
        }
      },
      {
        "label": "Encounter",
        "key": "encounter_2",
        "properties": {
          "sourcePostID": 1,
          "encounterDurationSec": 300
        }
      },
      {
        "label": "Traveller",
        "key": "traveller_1",
        "properties": {
          "entityName": "friend"
        }
      },
      {
        "label": "Traveller",
        "key": "traveller_2",
        "properties": {
          "entityName": "author"
        }
      },
      {
        "label": "BorderOfficer",
        "key": "officer_1",
        "properties": {}
      },
      {
        "label": "Airport",
        "key": "airport_1",
        "properties": {
          "entityName": "ШДГ"
        }
      },
    

In [ ]:
# JSON → Cypher генерация

In [31]:
import json
import re

# ---------- Helpers ----------

def strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def neo4j_local_name(name: str) -> str:
    """
    pcfr:entityName -> entityName
    full URI -> last fragment
    """
    if ":" in name:
        return name.split(":")[-1]
    if "#" in name:
        return name.split("#")[-1]
    if "/" in name:
        return name.rsplit("/", 1)[-1]
    return name

def cypher_literal(v):
    if v is None:
        return "null"
    if isinstance(v, bool):
        return "true" if v else "false"
    if isinstance(v, (int, float)):
        return str(v)
    s = str(v).replace("\\", "\\\\").replace("'", "\\'")
    return f"'{s}'"

# ---------- Parse JSON ----------

raw = strip_code_fences(answer)
root = json.loads(raw)

abox = root["payload"]
nodes = abox.get("nodes", [])
rels = abox.get("relationships", [])

# ---------- Generate Constraints ----------

labels = sorted({neo4j_local_name(n["label"]) for n in nodes})

cypher_statements = []

for label in labels:
    cypher_statements.append(
        f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{label}) REQUIRE n.key IS UNIQUE"
    )

# ---------- Generate Node Statements ----------

for node in nodes:
    label = neo4j_local_name(node["label"])
    key = node["key"]
    props = node.get("properties", {}) or {}

    set_parts = []
    for k, v in props.items():
        prop_key = neo4j_local_name(k)
        set_parts.append(f"n.{prop_key}={cypher_literal(v)}")

    set_clause = ""
    if set_parts:
        set_clause = " SET " + ", ".join(set_parts)

    stmt = f"MERGE (n:{label} {{key:{cypher_literal(key)}}}){set_clause}"
    cypher_statements.append(stmt)

# ---------- Generate Relationship Statements ----------

for rel in rels:
    fl = neo4j_local_name(rel["from"]["label"])
    fk = rel["from"]["key"]
    tl = neo4j_local_name(rel["to"]["label"])
    tk = rel["to"]["key"]
    rt = neo4j_local_name(rel["type"])

    stmt = (
        f"MATCH (a:{fl} {{key:{cypher_literal(fk)}}}), "
        f"(b:{tl} {{key:{cypher_literal(tk)}}}) "
        f"MERGE (a)-[:{rt}]->(b)"
    )

    cypher_statements.append(stmt)

print("LLM finish_reason:", root.get("finish_reason"))
print("LLM reason:", root.get("reason"))
print(f"Prepared {len(cypher_statements)} Cypher statements.")
print("\n--- First 10 statements ---")
for s in cypher_statements[:10]:
    print(s + ";")

LLM finish_reason: 🟢 complete
LLM reason: All entities and relationships from the text have been extracted, including two separate encounters for the two travellers, document checks, questioning topics, and location enrichment.
Prepared 84 Cypher statements.

--- First 10 statements ---
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Airport) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:BorderOfficer) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Cash) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:CashAmount) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:City) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Country) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:DocumentCheck) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Employment) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Encounter) REQUIRE n.key IS UNIQUE;
CREATE CONSTRAINT IF NOT EXISTS FOR (n:Fo

In [27]:
# print(cypher_statements)

In [11]:
# исполнение Cypher (каждая строка автономна)

In [32]:
from neo4j import GraphDatabase

URI = "neo4j://localhost:7688"
USER = "neo4j"
PASSWORD = "12345678"
DATABASE = "neo4j"

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

try:
    with driver.session(database=DATABASE) as session:
        for i, stmt in enumerate(cypher_statements, 1):
            session.run(stmt).consume()
    print(f"✅ Executed {len(cypher_statements)} statements.")
finally:
    driver.close()

✅ Executed 84 statements.
